# Module 3 Lab 1: Rescue a Messy Dataset

Krystal-Rose C. Uduma | CS 82A

Input: `messy_sales.csv` (Canvas course file, saved next to this notebook).

Portfolio: https://github.com/krystalroseuduma/cs82a-portfolio

## 1. Load and inspect the sales file

In [1]:
import pandas as pd

raw = pd.read_csv("messy_sales.csv")
sales = raw.copy()
print("Original shape:", sales.shape)
print(sales.head(10).to_string(index=False))
print("\nOriginal column types:")
print(sales.dtypes.to_string())

Original shape: (300, 6)
 order_id       date  product  price  qty   zip
     1254 04/06/2026      mug  11.36    4 60614
     1057 2026-05-30  charger  14.79    3 30303
     1150 05/06/2026      mug   5.50    1 10001
     1066 05/30/2026 notebook    NaN    1 98101
     1114 04/06/2026   webcam  45.59    4 90405
     1280 04/22/2026  pen set    NaN    4 30303
     1204 04/18/2026  charger   8.41    2  2134
     1249 2026-05-15 keyboard  75.04    4 90405
     1037 2026-05-19      mug   9.31    2 60614
     1177 2026-05-13      mug  10.11    3 90210

Original column types:
order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64


## 2. First observations

- The preview shows blank prices for orders 1066 and 1280.
- The date column mixes month/day/year and year-month-day strings.
- The ZIP column is numeric, and order 1204 shows 2134 where a five-digit ZIP should appear.

## 3. Count missing values

In [2]:
print(sales.isna().sum().to_string())

order_id     0
date         0
product      0
price       12
qty          0
zip          0


## 4. Fill missing prices with the median

In [3]:
missing_prices = int(sales["price"].isna().sum())
median_price = sales["price"].median()
print(f"Mean price before filling: ${sales['price'].mean():.2f}")
print(f"Median used to fill {missing_prices} prices: ${median_price:.2f}")
sales["price"] = sales["price"].fillna(median_price)
print("\nMissing values after filling:")
print(sales.isna().sum().to_string())

Mean price before filling: $56.07
Median used to fill 12 prices: $37.53

Missing values after filling:
order_id    0
date        0
product     0
price       0
qty         0
zip         0


## 5. Remove exact duplicate rows

In [4]:
before = len(sales)
duplicates = int(sales.duplicated().sum())
sales = sales.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicate rows: {duplicates}")
print(f"Rows: {before} -> {len(sales)}")
print("Shape after deduplication:", sales.shape)

Exact duplicate rows: 8
Rows: 300 -> 292
Shape after deduplication: (292, 6)


## 6. Restore five-character ZIP codes

In [5]:
old_zip = sales["zip"].astype(str)
sales["zip"] = old_zip.str.zfill(5)
padded_count = int(old_zip.ne(sales["zip"]).sum())
print("ZIP values padded:", padded_count)
print(sales.loc[old_zip.ne(sales["zip"]), ["order_id", "zip"]]
      .head().to_string(index=False))

ZIP values padded: 82
 order_id   zip
     1204 02134
     1108 02116
     1090 02116
     1074 02134
     1147 02134


## 7. Standardize dates

In [6]:
slash_dates = int(sales["date"].str.contains("/", regex=False).sum())
sales["date"] = pd.to_datetime(sales["date"], format="mixed").astype("datetime64[ns]")
print(f"Standardized {len(sales)} dates, including {slash_dates} slash-format dates.")
print(sales.dtypes.to_string())
print(sales[["order_id", "date"]].head().to_string(index=False))

Standardized 292 dates, including 146 slash-format dates.
order_id             int64
date        datetime64[ns]
product                str
price              float64
qty                  int64
zip                    str
 order_id       date
     1254 2026-04-06
     1057 2026-05-30
     1150 2026-05-06
     1066 2026-05-30
     1114 2026-04-06


## 8. Investigate negative quantities

I treated the three negative quantities as possible returns and kept the original signs. A return could legitimately reduce net sales, but the data dictionary gives no transaction type that confirms these rows are returns. I flagged the orders in the review table and kept the required six columns in the exported file. I would verify them against the return records before reporting a final sales total.

In [7]:
negative = sales["qty"].lt(0)
print("Original rows with negative quantity:")
print(sales.loc[negative].to_string(index=False))
print("\nHandled rows in the review table:")
review = sales.loc[negative, ["order_id", "product", "qty"]].copy()
review["review_status"] = "Possible return; verify"
print(review.to_string(index=False))

Original rows with negative quantity:
 order_id       date   product  price  qty   zip
     1140 2026-04-18    webcam  38.69   -5 98101
     1233 2026-05-09 desk lamp  22.64   -4 10001
     1025 2026-04-27  keyboard  47.30   -2 02116

Handled rows in the review table:
 order_id   product  qty           review_status
     1140    webcam   -5 Possible return; verify
     1233 desk lamp   -4 Possible return; verify
     1025  keyboard   -2 Possible return; verify


## 9. Save and verify the cleaned CSV

In [8]:
assert raw.shape == (300, 6)
assert missing_prices == 12 and duplicates == 8
assert sales.shape == (292, 6)
assert not sales.isna().any().any()
assert not sales.duplicated().any()
assert sales["zip"].str.fullmatch(r"\d{5}").all()
assert str(sales["date"].dtype) == "datetime64[ns]"
assert negative.sum() == 3

sales.to_csv("sales_clean.csv", index=False, date_format="%Y-%m-%d")
saved = pd.read_csv("sales_clean.csv", dtype={"zip": str})
assert saved.shape == sales.shape and list(saved) == list(sales)
assert saved["zip"].eq(sales["zip"]).all()
print("Saved sales_clean.csv:", saved.shape)
print("All checks passed. Three possible returns retained.")

Saved sales_clean.csv: (292, 6)
All checks passed. Three possible returns retained.


## 10. Cleaning Log

- Loaded 300 rows and six columns. I kept the raw CSV as the starting point for reruns.
- Filled 12 missing prices with the observed median of $37.53. The mean was $56.07, so the median is less sensitive to expensive items. Missing prices went from 12 to 0, and the row count stayed at 300.
- Removed eight exact repeats so those orders would only be counted once. The row count went from 300 to 292.
- Converted all 292 ZIP codes to text and restored leading zeros in 82 entries, so the ZIP codes can be used as five-character identifiers.
- Parsed all 292 dates, including 146 written with slashes, as datetimes. The CSV stores them consistently as year-month-day.
- Kept and flagged three possible returns (orders 1140, 1233, and 1025). The final export has 292 rows and six columns.

Turning possible returns into positive sales could inflate net revenue and lead the store to overorder inventory.